In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
from shapely.ops import nearest_points
import numpy as np
import matplotlib.pyplot as plt

# Veri yükleme fonksiyonları
def load_ticarethane_candidates(file_path):
    """Ticarethane aday noktalarını yükle"""
    df = pd.read_excel(file_path)
    gdf = gpd.GeoDataFrame(df)
    # String olarak gelen polygon verisini shapely geometrisine dönüştür
    gdf['geometry'] = gdf['geometry'].apply(lambda x: convert_polygon_str_to_geometry(x))
    return gdf

def load_sanayi_candidates(file_path):
    """Sanayi aday noktalarını yükle"""
    df = pd.read_excel(file_path)
    gdf = gpd.GeoDataFrame(df)
    gdf['geometry'] = gdf['geometry'].apply(lambda x: convert_polygon_str_to_geometry(x))
    return gdf

def load_roads(file_path):
    """Yol verilerini yükle"""
    df = pd.read_excel(file_path)
    gdf = gpd.GeoDataFrame(df)
    gdf['geometry'] = gdf['geometry'].apply(lambda x: convert_linestring_str_to_geometry(x))
    return gdf

def load_osb_cells(file_path):
    """OSB hücrelerini yükle"""
    df = pd.read_excel(file_path)
    gdf = gpd.GeoDataFrame(df)
    gdf['geometry'] = gdf['geometry'].apply(lambda x: convert_polygon_str_to_geometry(x))
    return gdf

def load_energy_permissions(file_path):
    """Enerji müsaadelerini yükle"""
    df = pd.read_excel(file_path)
    # Koordinat sütunlarını Point geometrisine dönüştür
    geometry = [Point(xy) for xy in zip(df['ENERJI_MUSAADE_X_KOORDINAT'], df['ENERJI_MUSAADE_Y_KOORDINAT'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry)
    return gdf

# String formatındaki geometri verilerini Shapely nesnelerine dönüştürme fonksiyonları
def convert_polygon_str_to_geometry(polygon_str):
    """POLYGON/MULTIPOLYGON string formatını shapely geometrisine dönüştür"""
    try:
        if "MULTIPOLYGON" in polygon_str:
            # MULTIPOLYGON formatı işleme
            coords_str = polygon_str.replace("MULTIPOLYGON((", "").replace("))", "")
            polygon_list = []
            
            # Her polygonu ayır
            polygon_parts = coords_str.split("), (")
            for part in polygon_parts:
                coords = []
                # Her koordinat çiftini ayır
                coord_pairs = part.replace("(", "").replace(")", "").split(", ")
                for pair in coord_pairs:
                    x, y = pair.split(" ")
                    coords.append((float(x), float(y)))
                polygon_list.append(Polygon(coords))
            
            return MultiPolygon(polygon_list)
        else:
            # POLYGON formatı işleme
            coords_str = polygon_str.replace("POLYGON((", "").replace("))", "")
            coords = []
            
            # Her koordinat çiftini ayır
            coord_pairs = coords_str.split(", ")
            for pair in coord_pairs:
                x, y = pair.split(" ")
                coords.append((float(x), float(y)))
            
            return Polygon(coords)
    except Exception as e:
        print(f"Polygon dönüştürme hatası: {e} - {polygon_str}")
        return None

def convert_linestring_str_to_geometry(linestring_str):
    """LINESTRING string formatını shapely geometrisine dönüştür"""
    try:
        coords_str = linestring_str.replace("LINESTRING (", "").replace(")", "")
        coords = []
        
        # Her koordinat çiftini ayır
        coord_pairs = coords_str.split(", ")
        for pair in coord_pairs:
            x, y = pair.split(" ")
            coords.append((float(x), float(y)))
        
        return LineString(coords)
    except Exception as e:
        print(f"LineString dönüştürme hatası: {e} - {linestring_str}")
        return None

# Mesafe hesaplama fonksiyonları
def calculate_distance_to_roads(cell_geometry, roads_gdf):
    """Hücrenin en yakın anayola (800m üzeri) olan mesafesini hesapla"""
    # Yalnızca 800m üzeri yollar
    main_roads = roads_gdf[roads_gdf['length'] >= 800]
    
    min_distance = float('inf')
    for _, road in main_roads.iterrows():
        distance = cell_geometry.distance(road.geometry)
        if distance < min_distance:
            min_distance = distance
    
    return min_distance

def calculate_distance_to_osb(cell_geometry, osb_gdf):
    """Hücrenin en yakın OSB hücresine olan mesafesini hesapla"""
    min_distance = float('inf')
    for _, osb_cell in osb_gdf.iterrows():
        distance = cell_geometry.distance(osb_cell.geometry)
        if distance < min_distance:
            min_distance = distance
    
    return min_distance

# Ana algoritma
def allocate_buildings(ticarethane_gdf, sanayi_gdf, roads_gdf, osb_gdf, energy_permissions_gdf, start_year, end_year, 
                       ticarethane_yearly_limit=(3, 5), sanayi_yearly_limit=1):
    """Belirlenen algoritma ile bina yerleşimi yap"""
    # Sonuç dataframe'i oluştur
    results = []
    
    # Enerji müsaadelerini abone grubu ve yıla göre grupla
    permission_groups = energy_permissions_gdf.groupby(['ENERJI_MUSAADE_ABONE_GRUBU', 'ENERJI_MUSAADE_ENERJILENDIRME_YILI'])
    
    # Ticarethane adayları için yollara olan mesafeleri hesapla
    print("Ticarethane hücreleri için anayollara olan mesafeleri hesaplıyorum...")
    ticarethane_gdf['road_distance'] = ticarethane_gdf['geometry'].apply(
        lambda geom: calculate_distance_to_roads(geom, roads_gdf))
    
    # Sanayi adayları için OSB'lere olan mesafeleri hesapla
    print("Sanayi hücreleri için OSB'lere olan mesafeleri hesaplıyorum...")
    sanayi_gdf['osb_distance'] = sanayi_gdf['geometry'].apply(
        lambda geom: calculate_distance_to_osb(geom, osb_gdf))
    
    # Mesafelere göre adayları sırala (en yakın önce)
    ticarethane_gdf = ticarethane_gdf.sort_values('road_distance')
    sanayi_gdf = sanayi_gdf.sort_values('osb_distance')
    
    # Kullanılmış hücreleri takip etmek için
    used_ticarethane_cells = set()
    used_sanayi_cells = set()
    
    # Yıllara göre işlem
    for year in range(start_year, end_year + 1):
        print(f"Yıl {year} için işlem yapılıyor...")
        
        # Ticarethane için yerleştirilmesi gereken bina sayısı (yıllık limit)
        if isinstance(ticarethane_yearly_limit, tuple):
            # Eğer aralık verildiyse
            ticarethane_count = np.random.randint(ticarethane_yearly_limit[0], ticarethane_yearly_limit[1] + 1)
        else:
            ticarethane_count = ticarethane_yearly_limit
        
        sanayi_count = sanayi_yearly_limit
        
        # 1. Önce enerji müsaadelerini yerleştir
        if ('ticarethane', year) in permission_groups.groups:
            ticarethane_permissions = permission_groups.get_group(('ticarethane', year))
            for _, permission in ticarethane_permissions.iterrows():
                # En yakın hücreyi bul
                closest_cell_idx = None
                closest_distance = float('inf')
                
                for idx, candidate in ticarethane_gdf.iterrows():
                    if candidate['id'] in used_ticarethane_cells:
                        continue
                    
                    distance = candidate.geometry.distance(permission.geometry)
                    if distance < closest_distance:
                        closest_distance = distance
                        closest_cell_idx = idx
                
                if closest_cell_idx is not None:
                    cell = ticarethane_gdf.loc[closest_cell_idx]
                    used_ticarethane_cells.add(cell['id'])
                    results.append({
                        'year': year,
                        'cell_id': cell['id'],
                        'building_type': 'BUYUK_TICARETHANE',
                        'source': 'energy_permission',
                        'x_coordinate': permission.geometry.x,
                        'y_coordinate': permission.geometry.y
                    })
                    ticarethane_count -= 1
        
        if ('sanayi', year) in permission_groups.groups:
            sanayi_permissions = permission_groups.get_group(('sanayi', year))
            for _, permission in sanayi_permissions.iterrows():
                # En yakın hücreyi bul
                closest_cell_idx = None
                closest_distance = float('inf')
                
                for idx, candidate in sanayi_gdf.iterrows():
                    if candidate['id'] in used_sanayi_cells:
                        continue
                    
                    distance = candidate.geometry.distance(permission.geometry)
                    if distance < closest_distance:
                        closest_distance = distance
                        closest_cell_idx = idx
                
                if closest_cell_idx is not None:
                    cell = sanayi_gdf.loc[closest_cell_idx]
                    used_sanayi_cells.add(cell['id'])
                    results.append({
                        'year': year,
                        'cell_id': cell['id'],
                        'building_type': 'BUYUK_SANAYI',
                        'source': 'energy_permission',
                        'x_coordinate': permission.geometry.x,
                        'y_coordinate': permission.geometry.y
                    })
                    sanayi_count -= 1
        
        # 2. Eğer hala eklenmesi gereken binalar varsa, yakınlık kriterlerine göre ekle
        # Ticarethane için
        for idx, candidate in ticarethane_gdf.iterrows():
            if ticarethane_count <= 0:
                break
                
            if candidate['id'] in used_ticarethane_cells:
                continue
                
            used_ticarethane_cells.add(candidate['id'])
            # Hücre merkezini hesapla
            centroid = candidate.geometry.centroid
            results.append({
                'year': year,
                'cell_id': candidate['id'],
                'building_type': 'BUYUK_TICARETHANE',
                'source': 'algorithm',
                'x_coordinate': centroid.x,
                'y_coordinate': centroid.y
            })
            ticarethane_count -= 1
        
        # Sanayi için
        for idx, candidate in sanayi_gdf.iterrows():
            if sanayi_count <= 0:
                break
                
            if candidate['id'] in used_sanayi_cells:
                continue
                
            used_sanayi_cells.add(candidate['id'])
            # Hücre merkezini hesapla
            centroid = candidate.geometry.centroid
            results.append({
                'year': year,
                'cell_id': candidate['id'],
                'building_type': 'BUYUK_SANAYI',
                'source': 'algorithm',
                'x_coordinate': centroid.x,
                'y_coordinate': centroid.y
            })
            sanayi_count -= 1
    
    return pd.DataFrame(results)

def main():
    # Veri dosya yolları
    ticarethane_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\ticaret_aday_noktalar.xlsx"
    sanayi_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\sanayi_aday_noktalar.xlsx"
    roads_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\izmir_tum_yol_kaldirim.xlsx"
    osb_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\izmir_osb_sonuclar.xlsx"
    energy_permissions_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\enerji_musaadeleri.xlsx"
    
    # Verileri yükle
    print("Veriler yükleniyor...")
    ticarethane_gdf = load_ticarethane_candidates(ticarethane_file)
    sanayi_gdf = load_sanayi_candidates(sanayi_file)
    roads_gdf = load_roads(roads_file)
    osb_gdf = load_osb_cells(osb_file)
    energy_permissions_gdf = load_energy_permissions(energy_permissions_file)
    
    # CRS kontrolü ve ayarı (eğer gerekirse)
    # Burada tüm veri setlerinin aynı projeksiyonda olduğunu varsayıyoruz
    
    # Bina yerleştirme algoritmasını çalıştır
    print("Bina yerleştirme algoritması çalıştırılıyor...")
    start_year = 2024  # Başlangıç yılı
    end_year = 2035    # Bitiş yılı
    
    results_df = allocate_buildings(
        ticarethane_gdf, 
        sanayi_gdf, 
        roads_gdf, 
        osb_gdf,
        energy_permissions_gdf,
        start_year,
        end_year
    )
    
    # Sonuçları kaydet
    output_file = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\point_load\buyuk_sanayi_ticarethane_yerlesimi.xlsx"
    results_df.to_excel(output_file, index=False)
    print(f"Sonuçlar {output_file} dosyasına kaydedildi.")
    

if __name__ == "__main__":
    main()

Veriler yükleniyor...
LineString dönüştürme hatası: too many values to unpack (expected 2) - POINT (27.07283152822559 38.48840728272754)
LineString dönüştürme hatası: too many values to unpack (expected 2) - POLYGON ((27.06986610654206 38.49951675205112, 27.06987270363028 38.49946976856645, 27.06959971689683 38.49949346072501, 27.0696050227306 38.49951330708525, 27.06963736032082 38.49954421617385, 27.06965523189616 38.49954348366784, 27.06986610654206 38.49951675205112))
LineString dönüştürme hatası: too many values to unpack (expected 2) - POLYGON ((27.06940090388415 38.49850239132762, 27.06938535755975 38.49846196291823, 27.06922751206995 38.49851357274379, 27.06924940443412 38.49855621318181, 27.06933114387812 38.49852735994372, 27.06940090388415 38.49850239132762))
LineString dönüştürme hatası: too many values to unpack (expected 2) - POLYGON ((27.06920000224682 38.4979375149101, 27.06886231246853 38.49800741806708, 27.06887998026223 38.49805541581554, 27.06910726583081 38.4980013

<Figure size 1200x600 with 0 Axes>